# First-principles VSi₂N₄: PBE+U+SOC starting notebook

This notebook starts from an **atomistic VSi₂N₄ structure** and performs an independent first-principles calculation using **ASE + GPAW**. It does **not** replace the DFT stage with an empirical tight-binding model.

## Reference target

Zhou *et al.* report VASP/PAW calculations with PBE-GGA, a 500 eV cutoff, a 25×25×1 static k-mesh, \(U_{eff}=3\) eV on V, and tight convergence criteria. Their paper reports \(a\approx2.88\) Å and uses HSE06 as an additional electronic-structure check. The present notebook implements the corresponding open-source PBE+U stage and then obtains SOC bands using GPAW's SOC machinery. The SOC step is a non-self-consistent diagonalization of the converged scalar-relativistic calculation, which is an important implementation distinction from a fully self-consistent SOC VASP calculation.

The reference paper: https://www.nature.com/articles/s41524-021-00632-3

**Computational philosophy:** every numerical result generated here is labeled `CALCULATED`; values from the paper are used only as validation targets.


## 0. Install the open-source stack

For a fresh Google Colab runtime, install ASE, GPAW, and GPAW's PAW datasets. The calculation is CPU-capable but the full 25×25×1 production run can be expensive for a free Colab runtime.

For development/debugging, use the smaller k-mesh in the configuration cell below. Switch to the reference 25×25×1 mesh only after the structure and convergence tests work.


In [ ]:
# Colab installation cell
!pip -q install ase gpaw
!gpaw install-data --yes

import sys, os, numpy as np
import ase, gpaw

print("Python:", sys.version)
print("ASE:", ase.__version__)
print("GPAW:", gpaw.__version__)


## 1. Imports and reproducibility configuration

The reference calculation uses a 500 eV plane-wave cutoff and 25×25×1 k-mesh for static calculations. GPAW's `PW()` mode accepts an explicit plane-wave cutoff in eV.

We deliberately separate `DEBUG` and `PRODUCTION` settings so that the expensive calculation is never launched accidentally.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from ase import Atoms
from ase.io import write, read
from ase.optimize import BFGS
from ase.constraints import FixAtoms
from ase.dft.kpoints import bandpath
from ase.units import Bohr

from gpaw import GPAW, PW, FermiDirac
from gpaw.poisson import PoissonSolver

# -----------------------------
# User-controlled configuration
# -----------------------------
DEBUG = True

A0 = 2.88                 # Angstrom; reference structural target
VACUUM = 13.0             # Angstrom; target separation along z
ECUT = 500                # eV; reference paper
U_EFF = 3.0               # eV; reference paper
NDEBUG = (5, 5, 1)        # fast test mesh
NPROD = (25, 25, 1)       # reference static mesh
KPTS = NDEBUG if DEBUG else NPROD

FMAX = 1e-3               # eV/Angstrom
ENERGY_TOL = 1e-7         # eV
SMEARING = 0.05           # eV; numerical occupation width

WORK = Path("VSi2N4_DFT")
WORK.mkdir(exist_ok=True)

print("k-mesh:", KPTS)
print("cutoff:", ECUT, "eV")
print("Ueff:", U_EFF, "eV")


## 2. Build the seven-layer VSi₂N₄ monolayer

The material has the septuple-layer sequence

\[
N-Si-N-V-N-Si-N,
\]

with a hexagonal primitive cell and one V, two Si, and four N atoms.

The exact relaxed internal coordinates are not given in the main text as a complete POSCAR. Therefore this notebook starts from a chemically sensible symmetric seed and **relaxes the atomic coordinates with DFT**. This avoids silently treating guessed internal coordinates as published data.

The initial seed is constructed from the experimentally/theoretically established hexagonal topology. The out-of-plane coordinate seed is intentionally only an initialization; the relaxed geometry is the quantity used for subsequent electronic calculations.


In [ ]:
def make_vsi2n4(a=A0, vacuum=VACUUM):
    # Hexagonal primitive lattice vectors.
    a1 = np.array([a, 0.0, 0.0])
    a2 = np.array([0.5*a, np.sqrt(3)*a/2, 0.0])

    # Initial fractional coordinates in the a1/a2 plane.
    # All atoms are initialized at the same in-plane registry (0,0)
    # except the symmetry-related N/Si sites generated below.
    #
    # A compact symmetric seed is used; z coordinates are relaxed by DFT.
    # Layer sequence: N-Si-N-V-N-Si-N.
    symbols = ["N", "Si", "N", "V", "N", "Si", "N"]

    # Fractional in-plane coordinates.  The two outer N/Si bilayers
    # are registered symmetrically around the V plane.
    xy = [
        (0.000, 0.000),  # outer N
        (1/3,   2/3),    # Si
        (0.000, 0.000),  # inner N
        (1/3,   2/3),    # V
        (0.000, 0.000),  # inner N
        (1/3,   2/3),    # Si
        (0.000, 0.000),  # outer N
    ]

    # Symmetric starting layer heights in Angstrom.
    z = np.array([-3.15, -2.25, -1.05, 0.0, 1.05, 2.25, 3.15])

    # Put the slab in a centered cell.
    Lz = 2*np.max(np.abs(z)) + vacuum
    zcart = z + Lz/2

    positions = []
    for (fx, fy), zz in zip(xy, zcart):
        positions.append(fx*a1 + fy*a2 + np.array([0.0, 0.0, zz]))

    cell = np.array([a1, a2, [0.0, 0.0, Lz]])

    atoms = Atoms(
        symbols=symbols,
        positions=positions,
        cell=cell,
        pbc=(True, True, True)
    )
    atoms.center(axis=2)
    return atoms

atoms = make_vsi2n4()
print(atoms)
print("cell =
", atoms.cell)
print("symbols =", atoms.get_chemical_symbols())

write(WORK / "VSi2N4_initial.xyz", atoms)


In [ ]:
# Visualize the initial atomistic structure.
from ase.visualize.plot import plot_atoms

fig, ax = plt.subplots(figsize=(6, 5))
plot_atoms(atoms, ax=ax, rotation=('90x,0y,0z'))
ax.set_title("Initial VSi$_2$N$_4$ seed")
plt.show()


## 3. Important structure sanity checks

Before running DFT, inspect the layer sequence, vacuum thickness, cell area, and minimum interatomic distances.

Because the internal seed is only an initialization, **do not interpret its bond lengths as final material properties**.


In [ ]:
from ase.geometry import get_distances

print("Chemical formula:", atoms.get_chemical_formula())
print("Cell lengths (A):", atoms.cell.lengths())
print("Cell angles (deg):", atoms.cell.angles())

# Pair-distance matrix under periodic boundary conditions.
D, _ = get_distances(atoms.positions, atoms.positions, cell=atoms.cell, pbc=atoms.pbc)
D = D + np.eye(len(atoms))*1e6
print("Minimum seeded distance (A):", D.min())


## 4. Spin-polarized PBE relaxation

The reference system is ferromagnetic and the electronic states near the Fermi level are strongly spin polarized. We therefore start with a spin-polarized PBE calculation.

A practical issue is that the published VASP calculation uses 500 eV, while a free Colab CPU calculation can be expensive. The code below is configured for the reference cutoff but keeps the k-mesh under the global `DEBUG` switch.

The initial V magnetic moment is set to approximately 1 \(\mu_B\), consistent with the reported order of magnitude for VSi₂N₄. This is an **initial magnetic guess**, not a fitted final result.


In [ ]:
# Initial magnetic moments: V carries the dominant moment.
initial_magmoms = np.zeros(len(atoms))
initial_magmoms[3] = 1.0
atoms.set_initial_magnetic_moments(initial_magmoms)

calc_relax = GPAW(
    mode=PW(ECUT),
    xc="PBE",
    kpts=KPTS,
    spinpol=True,
    occupations=FermiDirac(width=SMEARING),
    convergence={
        "energy": ENERGY_TOL,
        "density": 1e-5,
    },
    txt=str(WORK / "pbe_relax.txt"),
)

atoms.calc = calc_relax
E0 = atoms.get_potential_energy()
print("Initial PBE energy:", E0, "eV")
print("Initial magnetic moments:", atoms.get_magnetic_moments())


## 5. Geometry optimization

For a production study, the in-plane lattice should ideally be optimized by an equation-of-state/stress procedure rather than simply fixed at 2.88 Å. The reference paper's reported lattice scale is approximately 2.88 Å.

For the first reproduction pass we relax **internal coordinates at fixed \(a=2.88\) Å**. This separates geometry relaxation from lattice-constant optimization and makes the calculation easier to diagnose.

If the force relaxation fails, reduce the debug k-mesh/cutoff temporarily, diagnose convergence, then return to the reference settings.


In [ ]:
# Relax atomic positions at fixed in-plane cell.
opt = BFGS(atoms, trajectory=str(WORK / "pbe_relax.traj"),
            logfile=str(WORK / "pbe_bfgs.log"))
opt.run(fmax=FMAX, steps=150)

E_relaxed = atoms.get_potential_energy()
print("Relaxed PBE energy:", E_relaxed, "eV")
print("Final max force (eV/A):", np.max(np.linalg.norm(atoms.get_forces(), axis=1)))
print("Final magnetic moments:", atoms.get_magnetic_moments())

write(WORK / "VSi2N4_PBE_relaxed.xyz", atoms)


## 6. Save the converged scalar-relativistic PBE state

The `.gpw` file stores the converged GPAW state and is the starting point for the PBE+U and SOC stages.

This is particularly useful in Colab: if the runtime survives the calculation, later notebook cells do not need to repeat the entire relaxation.


In [ ]:
# Save the relaxed PBE state.
calc_relax.write(str(WORK / "VSi2N4_PBE_relaxed.gpw"), mode="all")
print("Saved:", WORK / "VSi2N4_PBE_relaxed.gpw")


## 7. PBE+U calculation

The paper uses \(U_{\mathrm{eff}}=3\) eV for V. GPAW implements the Dudarev-style effective-U correction; its documentation exposes this through the atomic setup syntax `':d,3.0'`.

This is therefore a genuine DFT+U calculation rather than an empirical tight-binding adjustment.


In [ ]:
# PBE + Ueff on V d states.
calc_u = GPAW(
    mode=PW(ECUT),
    xc="PBE",
    kpts=KPTS,
    spinpol=True,
    setups={"V": f":d,{U_EFF}"},
    occupations=FermiDirac(width=SMEARING),
    convergence={
        "energy": ENERGY_TOL,
        "density": 1e-5,
    },
    txt=str(WORK / "pbe_u.txt"),
)

atoms.calc = calc_u
E_u = atoms.get_potential_energy()
print("PBE+U energy:", E_u, "eV")
print("Magnetic moments:", atoms.get_magnetic_moments())

calc_u.write(str(WORK / "VSi2N4_PBE_U.gpw"), mode="all")


## 8. PBE+U band structure without SOC

The first electronic validation is the scalar-relativistic spin-polarized band structure.

The reference high-symmetry path is represented by

\[
\Gamma\rightarrow M\rightarrow K\rightarrow\Gamma.
\]

The paper reports a small direct gap near the K/K' valleys in the spin-up channel and a much larger spin-down gap. The exact numerical agreement is a validation test, not an input.


In [ ]:
# Obtain a band path through the 2D Brillouin zone.
bp = atoms.cell.bandpath("G-M-K-G", npoints=181)

calc_bs = GPAW(
    mode=PW(ECUT),
    xc="PBE",
    kpts=bp.kpts,
    spinpol=True,
    setups={"V": f":d,{U_EFF}"},
    occupations=FermiDirac(width=SMEARING),
    convergence={"bands": "all"},
    txt=str(WORK / "pbe_u_bands.txt"),
)

atoms.calc = calc_bs
atoms.get_potential_energy()

bs = calc_bs.band_structure()
bs.write(str(WORK / "VSi2N4_PBE_U_bandstructure.json"))


In [ ]:
# Plot scalar-relativistic PBE+U bands.
bs.plot(efermi=calc_bs.get_fermi_level(), emin=-4, emax=4,
        ylabel="Energy (eV)")
plt.title("VSi$_2$N$_4$: PBE+U, no SOC")
plt.show()


## 9. SOC bands from the converged PBE+U state

GPAW's documented SOC workflow takes a converged scalar-relativistic calculation and diagonalizes the SOC Hamiltonian in its Kohn--Sham basis. This gives SOC eigenvalues and spin characters.

This is an important methodological note: **the GPAW SOC call used here is non-self-consistent**, so the result is an independent open-source approximation to the reference workflow rather than a literal reproduction of a self-consistent VASP SOC run.

We will use the PBE+U ground state and calculate SOC bands along the same path.


In [ ]:
from gpaw.spinorbit import soc_eigenstates

# SOC diagonalization using the converged scalar-relativistic PBE+U state.
soc = soc_eigenstates(calc_u)
soc_eigs = soc.eigenvalues()

print("SOC eigenvalue array shape:", soc_eigs.shape)


## 10. SOC band path

For a band plot along \(\Gamma-M-K-\Gamma\), the scalar-relativistic calculation is evaluated on that path and the SOC Hamiltonian is diagonalized on the path.

The following cell creates a dedicated GPAW calculation on the band path, converges it, and then invokes the SOC diagonalization.


In [ ]:
calc_u_path = GPAW(
    mode=PW(ECUT),
    xc="PBE",
    kpts=bp.kpts,
    spinpol=True,
    setups={"V": f":d,{U_EFF}"},
    occupations=FermiDirac(width=SMEARING),
    convergence={"bands": "all"},
    txt=str(WORK / "pbe_u_path_for_soc.txt"),
)

atoms.calc = calc_u_path
atoms.get_potential_energy()
calc_u_path.write(str(WORK / "VSi2N4_PBE_U_path.gpw"), mode="all")

soc_path = soc_eigenstates(calc_u_path)
soc_path_eigs = soc_path.eigenvalues()

print("SOC path eigenvalues shape:", soc_path_eigs.shape)


In [ ]:
# Plot SOC eigenvalues.
x = bp.get_linear_kpoint_axis()[0]
energies = soc_path_eigs

# GPAW may return an array with k-points and bands.
plt.figure(figsize=(7, 5))
for n in range(energies.shape[1]):
    plt.plot(x, energies[:, n], linewidth=0.8)

for xpos in bp.get_linear_kpoint_axis()[1]:
    plt.axvline(xpos, linewidth=0.6)

plt.axhline(calc_u_path.get_fermi_level(), linewidth=0.8)
plt.xticks(bp.get_linear_kpoint_axis()[1], [r"$\Gamma$", "M", "K", r"$\Gamma$"])
plt.ylabel("Energy (eV)")
plt.title("VSi$_2$N$_4$: PBE+U + SOC")
plt.ylim(-3, 3)
plt.tight_layout()
plt.show()


## 11. Spin character of SOC states

The reference physics depends strongly on spin polarization. GPAW can return SOC eigenstates and spin character along the chosen quantization axis.

We inspect

\[
s_{nk}=\langle n\mathbf{k}|\sigma_z|n\mathbf{k}\rangle
\]

to quantify SOC-induced spin mixing.


In [ ]:
# Request SOC eigenstates and spin character.
soc_path_full = soc_eigenstates(calc_u_path)

try:
    spin_z = soc_path_full.spin_projections()
    print("Spin-projection array shape:", np.shape(spin_z))
except Exception as exc:
    print("Spin projection API differs in this GPAW version:", exc)


## 12. Basic band-gap / valley diagnostics

The paper reports approximately 0.24 eV direct gap in the relevant spin channel without SOC, and SOC valley splittings of approximately 102.3 meV (valence) and 27.3 meV (conduction).

The following cell is deliberately a diagnostic scaffold. Because the exact band indexing depends on the converged electron count and GPAW version, it does **not** hard-code a band number. It instead saves the band energies and k-point coordinates so the relevant K and K' extrema can be identified robustly.


In [ ]:
# Save numerical band data for reproducible post-processing.
np.save(WORK / "PBE_U_path_kpts.npy", np.array(bp.kpts))
np.save(WORK / "PBE_U_SOC_path_eigs.npy", np.array(soc_path_eigs))

print("Saved numerical SOC band data.")
print("Reference validation targets:")
print("  lattice scale ~ 2.88 A")
print("  PBE direct gap near K/K' ~ 0.24 eV")
print("  SOC valence valley splitting ~ 102.3 meV")
print("  SOC conduction valley splitting ~ 27.3 meV")


## 13. Optional: HSE06 validation

The paper uses HSE06 to check the electronic structure. This is **not** required for the first PBE+U+SOC milestone, and it can be substantially more expensive.

GPAW supports HSE06 in plane-wave mode. Run this only after the PBE+U calculation is stable.

The purpose is validation, not parameter fitting.


In [ ]:
# OPTIONAL / EXPENSIVE
RUN_HSE06 = False

if RUN_HSE06:
    calc_hse = GPAW(
        mode=PW(ECUT),
        xc="HSE06",
        kpts=KPTS,
        spinpol=True,
        occupations=FermiDirac(width=SMEARING),
        convergence={"energy": ENERGY_TOL, "density": 1e-5},
        txt=str(WORK / "hse06.txt"),
    )
    atoms.calc = calc_hse
    E_hse = atoms.get_potential_energy()
    print("HSE06 energy:", E_hse)
    calc_hse.write(str(WORK / "VSi2N4_HSE06.gpw"), mode="all")
else:
    print("HSE06 skipped.")


## 14. First strain point: 1.6%

Before building the full 0--2% strain sweep, test the central VQAH-region strain reported in the paper.

For biaxial strain

\[
\epsilon = \frac{a-a_0}{a_0},
\]

the in-plane cell is scaled uniformly. Internal coordinates are then relaxed with DFT+U.

This cell creates the 1.6% structure but does not yet launch a large production sweep.


In [ ]:
def strained_copy(atoms, strain):
    new = atoms.copy()
    cell = new.cell.copy()
    cell[0] *= (1.0 + strain)
    cell[1] *= (1.0 + strain)
    new.set_cell(cell, scale_atoms=True)
    return new

strain = 0.016
atoms_16 = strained_copy(atoms, strain)

calc_16 = GPAW(
    mode=PW(ECUT),
    xc="PBE",
    kpts=KPTS,
    spinpol=True,
    setups={"V": f":d,{U_EFF}"},
    occupations=FermiDirac(width=SMEARING),
    convergence={"energy": ENERGY_TOL, "density": 1e-5},
    txt=str(WORK / "strain_1p6_pbe_u.txt"),
)

atoms_16.calc = calc_16
atoms_16.get_potential_energy()

opt16 = BFGS(atoms_16,
              trajectory=str(WORK / "strain_1p6.traj"),
              logfile=str(WORK / "strain_1p6_bfgs.log"))
opt16.run(fmax=FMAX, steps=150)

calc_16.write(str(WORK / "VSi2N4_1p6_PBE_U.gpw"), mode="all")
write(WORK / "VSi2N4_1p6_relaxed.xyz", atoms_16)

print("1.6% strain calculation completed.")


## 15. Production roadmap after this notebook

Once the PBE+U+SOC stage is validated, the next notebook modules should add:

1. a 0--2% biaxial-strain sweep;
2. SOC band gaps at \(K\) and \(K'\);
3. orbital projections for V \(d_{z^2}\), \(d_{xy}\), and \(d_{x^2-y^2}\);
4. Wannierization of V-\(d\), Si-\(s,p\), and N-\(p\) manifolds;
5. dense \(501	imes501\) Berry-curvature interpolation;
6. Chern-number integration;
7. valley anomalous Hall conductivity;
8. valley anomalous Nernst conductivity;
9. frequency-dependent optical Hall conductivity and Kerr response;
10. quantitative reconstruction of Figs. 1--3.

The key methodological rule remains: **do not replace the DFT stage with a fitted effective Hamiltonian.** The effective model can be added later as an explanatory/validation layer, while the atomistic electronic structure remains the source of the parameters.


## References and implementation notes

- Zhou *et al.*, *npj Computational Materials* **7**, 160 (2021), “Sign-reversible valley-dependent Berry phase effects in 2D valley-half-semiconductors.”
- GPAW documentation: plane-wave calculations, DFT+U, and SOC.
- The reference paper reports PBE-GGA/PAW, 500 eV cutoff, 25×25×1 static k mesh, \(U_{eff}=3\) eV, and HSE06 validation.
- GPAW's DFT+U implementation uses the Dudarev effective-\(U\) form.
- GPAW's documented SOC workflow diagonalizes the SOC Hamiltonian using converged scalar-relativistic Kohn--Sham states.

**Reproducibility warning:** the initial internal coordinates in this notebook are a relaxation seed because the complete relaxed POSCAR was not supplied in the paper's main text. The relaxed DFT geometry, not the seed, is the scientific result.
